# Chapter 7 — Learning Signal and Ignoring Noise: Visualizing Neural Networks

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning, 2019)
**Chapter 7:** Learning Signal and Ignoring Noise: Introduction to Regularization and Batching

## Isi Chapter 7

1. Correlation summarization: ringkasan satu ide untuk semua pelajaran sebelumnya
2. Visualisasi yang disederhanakan: neural network sebagai balok LEGO
3. Melihat jaringan memprediksi (forward propagation)
4. Visualisasi dengan huruf: W dan l
5. Menghubungkan variabel dengan operasi
6. Pentingnya alat visualisasi

Chapter ini lebih banyak berisi cara berpikir daripada arsitektur baru. Karena itu
sebagian besar notebook ini berisi penjelasan, dan kode hanya dipakai untuk memastikan
bentuk (shape) matriks dan bahwa berbagai cara penulisan forward propagation memberi
hasil yang sama.

## 1. Correlation summarization

Ringkasan dari semua pelajaran sebelumnya:

> Neural network mencari korelasi langsung maupun tidak langsung antara layer input dan
> layer output, yang ditentukan oleh dataset input dan output.

Pada level lokal, setiap matriks bobot belajar mengkorelasikan layer inputnya dengan
apa yang seharusnya ada di layer output. Pada jaringan dengan banyak layer, apa yang
seharusnya dimiliki layer sebelumnya diperoleh dengan mengalikan kebutuhan layer
berikutnya dengan bobot di antaranya. Inilah yang disebut backpropagation. Jika
diibaratkan permainan telepon, setiap layer tahu apakah neuronnya perlu lebih tinggi
atau lebih rendah, lalu memperbarui bobotnya sendiri.

## 2. Visualisasi yang disederhanakan

Saya tidak perlu lagi memikirkan setiap bobot satu per satu. Yang penting adalah
bentuk (ukuran) setiap vektor dan matriks. Neural network bisa dianggap seperti balok
LEGO:

- **Vektor** adalah strip berisi node (misalnya `layer_1`).
- **Matriks** adalah kotak yang menghubungkan dua vektor (misalnya `weights_0_1`).

Bentuk matriks ditentukan oleh layer di sebelah kiri dan kanannya. Jika `layer_0`
berisi 3 node dan `layer_1` berisi 4 node, maka `weights_0_1` harus berbentuk 3 x 4.

Saya membuat fungsi kecil untuk menurunkan bentuk setiap matriks dari ukuran layer:

In [ ]:
import numpy as np

# Ukuran setiap layer pada jaringan tiga lapis dari chapter 6
layer_sizes = {'layer_0': 3, 'layer_1': 4, 'layer_2': 1}

# Bentuk matriks bobot = (jumlah node layer asal, jumlah node layer tujuan)
weight_shapes = {
    'weights_0_1': (layer_sizes['layer_0'], layer_sizes['layer_1']),
    'weights_1_2': (layer_sizes['layer_1'], layer_sizes['layer_2']),
}

for name, shape in weight_shapes.items():
    print(f'{name} : {shape}')

Hasilnya sesuai dengan aturan yang saya tulis: `weights_0_1` berbentuk 3 x 4 dan
`weights_1_2` berbentuk 4 x 1.

## 3. Melihat jaringan memprediksi

Saya membayangkan satu contoh dari dataset streetlight `[1, 0, 1]` mengalir dari
`layer_0` ke `layer_2` dalam empat langkah:

1. `layer_0` diisi dengan nilai input.
2. Empat weighted sum dihitung oleh `weights_0_1` (vector-matrix multiplication).
3. Hasilnya melewati relu, sehingga nilai negatif menjadi 0.
4. Weighted average terakhir oleh `weights_1_2` menghasilkan prediksi.

Saya menjalankan langkah ini dengan bobot acak untuk memastikan bentuk setiap
vektornya benar:

In [ ]:
def relu(x):
    return (x > 0) * x

np.random.seed(1)
layer_0 = np.array([[1, 0, 1]])                   # bentuk (1, 3)
weights_0_1 = 2 * np.random.random((3, 4)) - 1    # bentuk (3, 4)
weights_1_2 = 2 * np.random.random((4, 1)) - 1    # bentuk (4, 1)

layer_1_sum = layer_0.dot(weights_0_1)            # 4 weighted sum
layer_1 = relu(layer_1_sum)                        # nilai negatif -> 0
layer_2 = layer_1.dot(weights_1_2)                 # prediksi

print('layer_0 shape :', layer_0.shape)
print('layer_1 shape :', layer_1.shape)
print('layer_2 shape :', layer_2.shape)
print('layer_1 (sebelum relu):', np.round(layer_1_sum, 3))
print('layer_1 (sesudah relu):', np.round(layer_1, 3))
print('prediksi layer_2      :', np.round(layer_2, 3))

Dari hasil di atas, `layer_1` memiliki 4 nilai, dan nilai yang negatif berubah menjadi 0
setelah melewati relu. Prediksi `layer_2` adalah satu nilai. Ini sesuai dengan bentuk
yang saya hitung di bagian 2.

## 4. Visualisasi dengan huruf

Sebuah matriks diberi nama huruf kapital, dan sebuah vektor diberi nama huruf kecil:

- Matriks bobot: `W0`, `W1` (huruf W dengan nomor untuk membedakan)
- Vektor layer: `l0`, `l1`, `l2` (huruf l dengan nomor layer)

Dengan notasi ini, forward propagation bisa ditulis sebagai rumus:

- `l1 = relu(l0 W0)`: ambil `l0`, kalikan dengan matriks `W0`, lalu terapkan relu.
- `l2 = l1 W1`: ambil `l1`, kalikan dengan matriks `W1`.

Karena `l1` muncul sebagai input untuk `l2`, seluruh forward propagation bisa ditulis
dalam satu rumus: `l2 = relu(l0 W0) W1`.

## 5. Menghubungkan variabel dengan operasi

Notasi di atas memakai aturan sederhana: dua huruf yang ditulis berdampingan berarti
vector-matrix multiplication. Sebuah fungsi seperti relu bisa ditulis dengan nama
fungsinya di depan tanda kurung, mirip dengan kode Python.

Saya memeriksa bahwa tiga cara penulisan rumus di atas menghasilkan angka yang sama:

In [ ]:
np.random.seed(7)
l0 = np.random.random((1, 3))
W0 = 2 * np.random.random((3, 4)) - 1
W1 = 2 * np.random.random((4, 1)) - 1

# 1. Bentuk visualisasi / langkah demi langkah
l1_step = relu(l0.dot(W0))
l2_step = l1_step.dot(W1)

# 2. Bentuk rumus huruf: l2 = relu(l0 W0) W1 dalam satu baris
l2_formula = relu(l0.dot(W0)).dot(W1)

# 3. Bentuk kode dari chapter ini: layer_2 = relu(layer_0.dot(weights_0_1)).dot(weights_1_2)
layer_0, weights_0_1, weights_1_2 = l0, W0, W1
layer_2 = relu(layer_0.dot(weights_0_1)).dot(weights_1_2)

print('langkah demi langkah :', l2_step.ravel())
print('rumus huruf          :', l2_formula.ravel())
print('kode Python          :', layer_2.ravel())
print('ketiganya sama?      :', np.allclose(l2_step, l2_formula) and np.allclose(l2_formula, layer_2))

Ketiga bentuk penulisan memberi hasil yang sama. Yang berbeda hanya cara
menuliskannya, bukan perhitungannya.

## 6. Pentingnya alat visualisasi

Chapter ini menekankan bahwa cara berpikir dan bahasa yang sama dibutuhkan untuk
membaca arsitektur baru. Sebuah arsitektur mengatur bagaimana sinyal mengalir melalui
jaringan. Arsitektur yang baik membuat korelasi mudah ditemukan, dan arsitektur yang
sangat baik juga menyaring noise agar tidak terjadi overfitting.

Karena itu, saya perlu bisa membaca setiap arsitektur dengan tiga cara sekaligus:
gambar, rumus huruf, dan kode. Jika salah satu tidak jelas, saya perlu kembali ke
bagian sebelumnya sebelum melanjutkan.

## Key Ideas — Chapter 7

- Correlation summarization: neural network mencari dan membuat korelasi antara layer input dan output.
- Backpropagation adalah cara layer berikutnya memberi tahu layer sebelumnya sinyal apa yang dibutuhkan.
- Neural network seperti LEGO: setiap bagian adalah vektor atau matriks, dan bentuknya ditentukan oleh layer di sekitarnya.
- Arsitektur adalah susunan layer dan matriks; arsitektur mengatur bagaimana sinyal mengalir.
- Forward propagation bisa ditulis sebagai gambar, rumus huruf (`l2 = relu(l0 W0) W1`), atau kode.

## Kesimpulan

Chapter 7 tidak menambah teknik pelatihan baru. Chapter ini memberi bahasa dan cara
berpikir untuk membaca serta membangun arsitektur: ringkasan korelasi, bentuk matriks
yang ditentukan oleh layer, dan notasi huruf untuk forward propagation. Chapter
berikutnya menggunakan bahasa ini untuk membahas regularization dan batching.